# Process HA DMS data from Yu et al.

Import Python modules

In [1]:
import os
import pandas as pd
from collections import defaultdict
from Bio import SeqIO
import subprocess

In [2]:
import yaml

_config_dir = os.path.dirname(os.path.abspath("../config.yaml"))
with open("../config.yaml") as _f:
    _config = yaml.safe_load(_f)
data_dir = os.path.normpath(os.path.join(_config_dir, _config["data_dir"]))

Align the sequence from the DMS experiment to the reference sequence used to build the H3 tree.

In [3]:
# Read in the sequence of the reference HA used to build the tree
ref_fasta = os.path.join(data_dir, 'HA/H3/curated_reference.fasta')
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq.translate())
print(len(ref_aa_seq))

# Read in the HA sequence from the DMS experiment
numbering_df = pd.read_csv('../data/dms_data/Yu_HA/site_numbering_map.csv')
aa_seq = ''.join(list(numbering_df['sequential_wt']))

# Save sequences to a FASTA file for alignment
output_dir = '../results/dms_data/Yu_HA/'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)
unaligned_fasta = os.path.join(output_dir, 'unaligned.fasta')
if not os.path.isfile(unaligned_fasta):
    with open(unaligned_fasta, 'w') as f:
        f.write('>reference\n')
        f.write(ref_aa_seq + '\n')
        f.write('>dms\n')
        f.write(aa_seq + '\n')

# Align the sequences using MUSCLE
aligned_fasta = f'{output_dir}/aligned.fasta'
if not os.path.isfile(aligned_fasta):
    cmd = ['muscle', '-align', unaligned_fasta, '-output', aligned_fasta]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)

567


Read in the aligned sequences and determine numbering scheme

In [4]:
# Read in the aligned sequences
seqs_dict = defaultdict(list)
aligned_records = list(SeqIO.parse(aligned_fasta, 'fasta'))
for record in aligned_records:
    seqs_dict[record.id] = str(record.seq)

aligned_ref_seq = seqs_dict['reference']
aligned_dms_seq = seqs_dict['dms']

# Compute percent identity
n_sites_to_compare = 0
n_identical = 0
for (alignment_index, (ref_aa, dms_aa)) in enumerate(zip(aligned_ref_seq, aligned_dms_seq), 1):
    if ref_aa != '-' and dms_aa != '-':
        n_sites_to_compare += 1
        if ref_aa == dms_aa:
            n_identical += 1

print(n_sites_to_compare, n_identical, n_identical / n_sites_to_compare)

# Determine numbering scheme
numbering_dict = defaultdict(list)
assert len(aligned_ref_seq) == len(aligned_dms_seq)
alignment_length = len(aligned_ref_seq)
for (seq_id, seq) in [('tree_reference_site', aligned_ref_seq), ('sequential_site', aligned_dms_seq)]:
    seq_index = 1
    for (alignment_index, aa) in enumerate(seq, 1):
        if aa != '-':
            numbering_dict['alignment_index'].append(alignment_index)
            numbering_dict[f'seq_id'].append(seq_id)
            numbering_dict[f'seq_index'].append(seq_index)
            numbering_dict[f'seq_aa'].append(aa)
            seq_index += 1

alignment_numbering_df = (
    pd.DataFrame(numbering_dict)
    .pivot(index='alignment_index', columns='seq_id', values='seq_index')
    .reset_index()
    .rename_axis(None, axis=1)
    .dropna()
    [['sequential_site', 'tree_reference_site', ]]
    .merge(numbering_df)
)
alignment_numbering_df['sequential_site'] = alignment_numbering_df['sequential_site'].astype(int)
alignment_numbering_df['tree_reference_site'] = alignment_numbering_df['tree_reference_site'].astype(int)
del alignment_numbering_df['region']
alignment_numbering_df.head()

504 425 0.8432539682539683


,sequential_site,tree_reference_site,reference_site,sequential_wt,rbs_region
0,1,17,1,Q,outside RBS
1,2,18,2,K,outside RBS
2,3,19,3,I,outside RBS
3,4,20,4,P,outside RBS
4,5,21,5,G,outside RBS


Read in the DMS data and add columns that map from DMS numbering to alignment numbering.

In [5]:
# Read in DMS data and merge with numbering data and fitness data
f = '../data/dms_data/Yu_HA/Phenotypes.csv'
ha_dms_data = (
    pd.read_csv(f)
    .rename(
        columns={
            'MDCKSIAT1 cell entry' : 'dms_effect',
            'sera escape' : 'sera_escape',
            'pH stability' : 'pH_stability',
            'wildtype' : 'wt_aa',
            'mutant' : 'mut_aa',
            'nt changes to codon' : 'n_nt_changes'
        }
    )
    .merge(alignment_numbering_df, on='sequential_site', validate='many_to_one')
    # drop rows where dms_effect is NA
    .dropna(subset=['dms_effect'])
)
del ha_dms_data['region']
del ha_dms_data['rbs_region']
ha_dms_data.to_csv('../results/dms_data/Yu_HA/processed_dms_data.csv', index=False)
print("Number of mutations with processed data", len(ha_dms_data))
ha_dms_data.head()

Number of mutations with processed data 9738


,site,wt_aa,mut_aa,sera_escape,dms_effect,pH_stability,sequential_site,n_nt_changes,tree_reference_site,reference_site,sequential_wt
0,1,Q,A,0.08825,-0.1226,0.004237,1,2,17,1,Q
1,1,Q,C,0.01779,-0.5732,-0.014300,1,3,17,1,Q
2,1,Q,D,-0.05395,0.2550,-0.021900,1,2,17,1,Q
3,1,Q,E,-0.01963,0.2941,0.006890,1,1,17,1,Q
4,1,Q,F,-0.16350,-0.7141,-0.001402,1,3,17,1,Q
